# Network Anomaly Detection: UNSW-NB15 Dataset

Plan:

1. Load and inspect the dataset.
2. Remove duplicates and create stratified train, validation, and test splits.
3. Prepare Tier 1 and Tier 2 datasets and fit feature preprocessing.
4. Train the Tier 1 baseline and evaluate it on validation data.
5. Tune the decision threshold and inspect recall by attack family.
6. Compare a LightGBM Tier 1 model and train a Tier 2 attack-family classifier.

## 1. Setup and Data Loading

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("../data/UNSW-NB15/UNSW-NB15.csv")
PREPROCESSOR_PATH = Path("preprocessor.joblib")
RANDOM_STATE = 42
TARGET_FPR = 0.01


In [2]:
df = pd.read_csv(DATA_PATH)
df.head()


,proto,state,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,service,...,ct_flw_http_mthd,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat,Label
0,Udp,Con,0.036133,528,304,31,29,0,0,-,...,0,0,0,2,4,1,1,2,Normal,0
1,Udp,Con,0.001119,146,178,31,29,0,0,Dns,...,0,0,0,12,8,2,1,1,Normal,0
2,Udp,Con,0.001209,132,164,31,29,0,0,Dns,...,0,0,0,6,9,1,1,1,Normal,0
3,Udp,Con,0.001169,146,178,31,29,0,0,Dns,...,0,0,0,7,9,1,1,1,Normal,0
4,Udp,Con,0.078339,568,312,31,29,0,0,-,...,0,0,0,2,4,1,1,2,Normal,0


## 2. Exploring the Raw Data

Inspecting the schema, target balance, missing values, and duplicate rows before modifying the dataset.

In [3]:
print("Rows and columns:", df.shape)
df.info()


Rows and columns: (2540043, 38)
<class 'pandas.DataFrame'>
RangeIndex: 2540043 entries, 0 to 2540042
Data columns (total 38 columns):
 #   Column            Dtype  
---  ------            -----  
 0   proto             str    
 1   state             str    
 2   dur               float64
 3   sbytes            int64  
 4   dbytes            int64  
 5   sttl              int64  
 6   dttl              int64  
 7   sloss             int64  
 8   dloss             int64  
 9   service           str    
 10  Sload             float64
 11  Spkts             int64  
 12  Dpkts             int64  
 13  swin              int64  
 14  dwin              int64  
 15  stcpb             int64  
 16  dtcpb             int64  
 17  smeansz           int64  
 18  dmeansz           int64  
 19  trans_depth       int64  
 20  res_bdy_len       int64  
 21  Sjit              float64
 22  Djit              float64
 23  Sintpkt           float64
 24  Dintpkt           float64
 25  tcprtt            float6

In [4]:
df.columns.tolist()

['proto',
 'state',
 'dur',
 'sbytes',
 'dbytes',
 'sttl',
 'dttl',
 'sloss',
 'dloss',
 'service',
 'Sload',
 'Spkts',
 'Dpkts',
 'swin',
 'dwin',
 'stcpb',
 'dtcpb',
 'smeansz',
 'dmeansz',
 'trans_depth',
 'res_bdy_len',
 'Sjit',
 'Djit',
 'Sintpkt',
 'Dintpkt',
 'tcprtt',
 'is_sm_ips_ports',
 'ct_state_ttl',
 'ct_flw_http_mthd',
 'is_ftp_login',
 'ct_ftp_cmd',
 'ct_srv_src',
 'ct_srv_dst',
 'ct_src_dport_ltm',
 'ct_dst_sport_ltm',
 'ct_dst_src_ltm',
 'attack_cat',
 'Label']

In [5]:
print("Binary label counts:")
print(df["Label"].value_counts())

print("\nAttack category distribution:")
category_counts = df["attack_cat"].value_counts()
category_distribution = pd.DataFrame({
    "Count": category_counts,
    "Percent": df["attack_cat"].value_counts(normalize=True) * 100,
})
category_distribution


Binary label counts:
Label
0    2218760
1     321283
Name: count, dtype: int64

Attack category distribution:


,Count,Percent
attack_cat,,
Normal,2218760,87.351277
Generic,215481,8.483360
Exploits,44525,1.752923
Fuzzers,24246,0.954551
Dos,16353,0.643808
Reconnaissance,13987,0.550660
Analysis,2677,0.105392
Backdoors,2329,0.091691
Shellcode,1511,0.059487


In [6]:
print("Missing values per column:")
print(df.isna().sum().sort_values(ascending=False))
print("\nDuplicate rows:", df.duplicated().sum())

Missing values per column:
proto               0
ct_flw_http_mthd    0
Sjit                0
Djit                0
Sintpkt             0
Dintpkt             0
tcprtt              0
is_sm_ips_ports     0
ct_state_ttl        0
is_ftp_login        0
state               0
ct_ftp_cmd          0
ct_srv_src          0
ct_srv_dst          0
ct_src_dport_ltm    0
ct_dst_sport_ltm    0
ct_dst_src_ltm      0
attack_cat          0
res_bdy_len         0
trans_depth         0
dmeansz             0
smeansz             0
dur                 0
sbytes              0
dbytes              0
sttl                0
dttl                0
sloss               0
dloss               0
service             0
Sload               0
Spkts               0
Dpkts               0
swin                0
dwin                0
stcpb               0
dtcpb               0
Label               0
dtype: int64

Duplicate rows: 571625


## 3. Removing Duplicates and Inspecting Targets

Removing exact duplicate rows before splitting so identical flows cannot appear in multiple splits. Cross-checking binary labels against attack categories after deduplication.

In [7]:
df = df.drop_duplicates().reset_index(drop=True)

print("After deduplication:", df.shape)
print("Duplicates:", df.duplicated().sum())


After deduplication: (1968418, 38)
Duplicates: 0


In [8]:
print("\nBinary labels:")
print(df["Label"].value_counts())

print("\nAttack categories:")
print(df["attack_cat"].value_counts())

print("\nLabel consistency:")
print(pd.crosstab(df["attack_cat"], df["Label"]))


Binary labels:
Label
0    1885530
1      82888
Name: count, dtype: int64

Attack categories:
attack_cat
Normal            1885530
Exploits            27187
Fuzzers             20347
Generic             15895
Reconnaissance       9076
Dos                  5262
Analysis             1809
Backdoors            1710
Shellcode            1432
Worms                 170
Name: count, dtype: int64

Label consistency:
Label                 0      1
attack_cat                    
Analysis              0   1809
Backdoors             0   1710
Dos                   0   5262
Exploits              0  27187
Fuzzers               0  20347
Generic               0  15895
Normal          1885530      0
Reconnaissance        0   9076
Shellcode             0   1432
Worms                 0    170


## 4. Create and Verify the Data Splits

70% Training
15% Validation
15% Testing

Stratifying both splits by `attack_cat` to preserve attack-family proportions.

Then we use validation data for model and threshold analysis. **We are going to reserve test data for final validation.**

In [9]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=df["attack_cat"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["attack_cat"],
)


In [10]:
# Distribution of attack categories in train, validation, and test sets

print("\nTrain:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

distribution = pd.DataFrame({
    "Train": train_df["attack_cat"].value_counts(),
    "Validation": val_df["attack_cat"].value_counts(),
    "Test": test_df["attack_cat"].value_counts()
})

print(distribution)


Train: (1377892, 38)
Validation: (295263, 38)
Test: (295263, 38)
                  Train  Validation    Test
attack_cat                                 
Normal          1319871      282829  282830
Exploits          19031        4078    4078
Fuzzers           14243        3052    3052
Generic           11127        2384    2384
Reconnaissance     6353        1361    1362
Dos                3683         789     790
Analysis           1266         272     271
Backdoors          1197         257     256
Shellcode          1002         215     215
Worms               119          26      25


In [11]:
print("Training attack percentage:", train_df["Label"].mean() * 100)
print("Validation attack percentage:", val_df["Label"].mean() * 100)


Training attack percentage: 4.210852519645951
Validation attack percentage: 4.211160897233992


## 5. Preparing Features and Targets

We exclude both target columns from the features. Tier 1 uses every flow and the binary `Label`. Tier 2 will use attack flows only to predict `attack_cat`.

### 5.1 Tier 1: Normal vs. Attack

In [12]:
feature_columns = [
    col for col in df.columns if col not in ["Label", "attack_cat"]
]

X_train = train_df[feature_columns]
y_train = train_df["Label"]

X_val = val_df[feature_columns]
y_val = val_df["Label"]

X_test = test_df[feature_columns]
y_test = test_df["Label"]

print("Train features / target:", X_train.shape, y_train.shape)
print("Validation features / target:", X_val.shape, y_val.shape)
print("Test features / target:", X_test.shape, y_test.shape)
print("Feature columns:", feature_columns)


Train features / target: (1377892, 36) (1377892,)
Validation features / target: (295263, 36) (295263,)
Test features / target: (295263, 36) (295263,)
Feature columns: ['proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm']


### 5.2 Tier 2: Attack Family

In [13]:
attack_train = train_df[train_df["Label"] == 1].copy()
attack_val = val_df[val_df["Label"] == 1].copy()
attack_test = test_df[test_df["Label"] == 1].copy()

X_train_tier2 = attack_train[feature_columns]
y_train_tier2 = attack_train["attack_cat"]

X_val_tier2 = attack_val[feature_columns]
y_val_tier2 = attack_val["attack_cat"]

X_test_tier2 = attack_test[feature_columns]
y_test_tier2 = attack_test["attack_cat"]

## 6. Preprocessing

Decided to one-hot encode `proto`, `state`, and `service`, then standardize the remaining features. 

We fit the preprocessor on Tier 1 training data only, then reuse it to transform validation and test data. The Tier 2 datasets above remain unchanged.

In [14]:
categorical_features = ["proto", "state", "service"]
numerical_features = [
    col for col in X_train.columns if col not in categorical_features
]

print("Categorical features:", categorical_features)
print("Number of numerical features:", len(numerical_features))


Categorical features: ['proto', 'state', 'service']
Number of numerical features: 33


In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=True),
            categorical_features,
        ),
        ("numerical", StandardScaler(), numerical_features),
    ]
)


In [16]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Original training shape:", X_train.shape)
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Processed test shape:", X_test_processed.shape)


Original training shape: (1377892, 36)
Processed training shape: (1377892, 197)
Processed validation shape: (295263, 197)
Processed test shape: (295263, 197)


In [17]:
# Feature names after preprocessing

feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print(feature_names[:40])


Number of processed features: 197
['categorical__proto_3Pc' 'categorical__proto_A/N'
 'categorical__proto_Aes-Sp3-D' 'categorical__proto_Any'
 'categorical__proto_Argus' 'categorical__proto_Aris'
 'categorical__proto_Arp' 'categorical__proto_Ax.25'
 'categorical__proto_Bbn-Rcc' 'categorical__proto_Bna'
 'categorical__proto_Br-Sat-Mon' 'categorical__proto_Cbt'
 'categorical__proto_Cftp' 'categorical__proto_Chaos'
 'categorical__proto_Compaq-Peer' 'categorical__proto_Cphb'
 'categorical__proto_Cpnx' 'categorical__proto_Crtp'
 'categorical__proto_Crudp' 'categorical__proto_Dcn'
 'categorical__proto_Ddp' 'categorical__proto_Ddx'
 'categorical__proto_Dgp' 'categorical__proto_Egp'
 'categorical__proto_Eigrp' 'categorical__proto_Emcon'
 'categorical__proto_Encap' 'categorical__proto_Esp'
 'categorical__proto_Etherip' 'categorical__proto_Fc'
 'categorical__proto_Fire' 'categorical__proto_Ggp'
 'categorical__proto_Gmtp' 'categorical__proto_Gre'
 'categorical__proto_Hmp' 'categorical__proto_I-Nl

### 6.1 Saving Preprocessor

In [18]:
joblib.dump(preprocessor, PREPROCESSOR_PATH)


['preprocessor.joblib']

## 7. Tier 1 Baseline: Logistic Regression

Decided to use logistic regression with balanced class weights.
 
We generated validation predictions and attack probabilities for evaluation and threshold selection.

In [19]:
tier1_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=RANDOM_STATE,
)
tier1_model.fit(X_train_processed, y_train)

y_val_pred = tier1_model.predict(X_val_processed)
y_val_prob = tier1_model.predict_proba(X_val_processed)[:, 1]


## 8. Evaluating Threshold

Looking at validation classification metrics and the confusion matrix with the default threshold. 

The average precision summarizes the precision–recall curve using the predicted probabilities.

In [20]:
print(classification_report(
    y_val,
    y_val_pred,
    target_names=["Normal", "Attack"],
))

cm = confusion_matrix(y_val, y_val_pred)
print("Confusion matrix (rows: actual, columns: predicted; Normal, Attack):")
print(cm)


              precision    recall  f1-score   support

      Normal       1.00      0.99      0.99    282829
      Attack       0.75      1.00      0.86     12434

    accuracy                           0.99    295263
   macro avg       0.87      0.99      0.92    295263
weighted avg       0.99      0.99      0.99    295263

Confusion matrix (rows: actual, columns: predicted; Normal, Attack):
[[278674   4155]
 [     4  12430]]


In [21]:
# Evaluation metrics

tn, fp, fn, tp = cm.ravel()

precision = precision_score(y_val, y_val_pred)
recall = recall_score(y_val, y_val_pred)
f1 = f1_score(y_val, y_val_pred)

false_positive_rate = fp / (fp + tn)

average_precision = average_precision_score(
    y_val,
    y_val_prob
)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("False Positive Rate:", false_positive_rate)
print("Average precision:", average_precision)


Precision: 0.7494724148326801
Recall: 0.9996783014315587
F1: 0.8566801061373582
False Positive Rate: 0.014690855605330429
Average precision: 0.938447837054874


## 9. Adjusting the Threshold

### 9.1 Selecting a Threshold Under the False Positive Limit

Trying to choose the threshold with the highest validation recall among those with a false positive rate of at most 1%.

This limit is measured on validation data.

In [22]:
fpr, tpr, thresholds = roc_curve(
    y_val,
    y_val_prob,
    drop_intermediate=False,
)

valid_indices = np.where(fpr <= TARGET_FPR)[0]
best_idx = valid_indices[np.argmax(tpr[valid_indices])]
best_threshold = thresholds[best_idx]

print("Chosen threshold:", best_threshold)
print("Validation FPR:", fpr[best_idx])
print("Validation recall:", tpr[best_idx])

Chosen threshold: 0.9624737740077962
Validation FPR: 0.009998974645457149
Validation recall: 0.9152324272156989


### 9.2 Evaluate the Selected Threshold

In [23]:
y_val_tuned = (
    y_val_prob >= best_threshold
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_val,
    y_val_tuned,
    labels=[0, 1]
).ravel()

precision = precision_score(y_val, y_val_tuned, zero_division=0)
recall = recall_score(y_val, y_val_tuned)
f1 = f1_score(y_val, y_val_tuned)

false_positive_rate = fp / (fp + tn)

print("Chosen threshold:", best_threshold)
print()
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)
print()
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("FPR:", false_positive_rate)


Chosen threshold: 0.9624737740077962

TN: 280001
FP: 2828
FN: 1054
TP: 11380

Precision: 0.8009572072072072
Recall: 0.9152324272156989
F1: 0.8542902184520682
FPR: 0.009998974645457149


### 9.3 Compare different Thresholds

In [24]:
def evaluate_threshold(threshold):
    predictions = (
        y_val_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val,
        predictions,
        labels=[0, 1]
    ).ravel()

    return {
        "Threshold": threshold,
        "Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "Recall": recall_score(y_val, predictions),
        "F1": f1_score(y_val, predictions),
        "FPR": fp / (fp + tn),
        "False Positives": fp,
        "Missed Attacks": fn
    }

results = []

for threshold in [0.5, 0.6, 0.7, 0.8, 0.9]:
    results.append(evaluate_threshold(threshold))

results.append(evaluate_threshold(best_threshold))

results_df = pd.DataFrame(results)

print(results_df.round(4).to_string(index=False))


 Threshold  Precision  Recall     F1    FPR  False Positives  Missed Attacks
    0.5000     0.7495  0.9997 0.8567 0.0147             4155               4
    0.6000     0.7493  0.9985 0.8561 0.0147             4153              19
    0.7000     0.7491  0.9962 0.8552 0.0147             4148              47
    0.8000     0.7497  0.9943 0.8549 0.0146             4127              71
    0.9000     0.7528  0.9879 0.8545 0.0143             4033             151
    0.9625     0.8010  0.9152 0.8543 0.0100             2828            1054


## 10. Detection by Attack Family

We are comparing validation recall for each attack family at the default and tuned thresholds. 

This is reviewing missed attacks as well as the overall false positive rate to understand which threshold was best.

In [25]:
audit_df = val_df[["Label", "attack_cat"]].copy()

# Add the model's predicted probabilities
audit_df["attack_probability"] = y_val_prob

# Predictions using the original threshold
audit_df["pred_original"] = (
    y_val_prob >= 0.50
).astype(int)

# Predictions using the tuned threshold
audit_df["pred_tuned"] = (
    y_val_prob >= best_threshold
).astype(int)

# Only examine actual attacks
attacks_only = audit_df[
    audit_df["Label"] == 1
].copy()

# Comparing detection across attack families

family_results = attacks_only.groupby(
    "attack_cat"
).agg(
    Total=("Label", "size"),
    Detected_Original=("pred_original", "sum"),
    Detected_Tuned=("pred_tuned", "sum")
)

family_results["Missed_Original"] = (
    family_results["Total"] - family_results["Detected_Original"]
)

family_results["Missed_Tuned"] = (
    family_results["Total"] - family_results["Detected_Tuned"]
)

family_results["Recall_Original"] = (
    family_results["Detected_Original"] / family_results["Total"]
)

family_results["Recall_Tuned"] = (
    family_results["Detected_Tuned"] / family_results["Total"]
)

print(family_results.round(4).to_string())


                Total  Detected_Original  Detected_Tuned  Missed_Original  Missed_Tuned  Recall_Original  Recall_Tuned
attack_cat                                                                                                            
Analysis          272                272             248                0            24           1.0000        0.9118
Backdoors         257                257             237                0            20           1.0000        0.9222
Dos               789                788             718                1            71           0.9987        0.9100
Exploits         4078               4077            3886                1           192           0.9998        0.9529
Fuzzers          3052               3050            2491                2           561           0.9993        0.8162
Generic          2384               2384            2353                0            31           1.0000        0.9870
Reconnaissance   1361               1361        

## 11. Trying Different Model: LightGBM

We are training a second Tier 1 classifier on the same splits. 

LightGBM is a Gradient Boosting Decision Tree that uses the original numeric features and categorical features rather than the encoded and scaled matrices used by logistic regression.

This does requires `lightgbm` to be installed in order to run this code locally.

### 11.1 Preparing Categorical Features

Here we are copying the training and validation features, then learning each category vocabulary from the training data only.

Validation data values outside those vocabularies become missing categorical values.

In [26]:
import lightgbm as lgb

X_train_lgb = X_train.copy()
X_val_lgb = X_val.copy()

for col in categorical_features:
    categories = X_train_lgb[col].astype("category").cat.categories
    X_train_lgb[col] = pd.Categorical(X_train_lgb[col], categories=categories)
    X_val_lgb[col] = pd.Categorical(X_val_lgb[col], categories=categories)


### 11.2 Train the Model

In [27]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=50,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)
lgb_model.fit(
    X_train_lgb,
    y_train,
    categorical_feature=categorical_features,
)

lgb_probs = lgb_model.predict_proba(X_val_lgb)[:, 1]


### 11.3 Evaluating the Default Threshold

In [28]:
lgb_default_preds = (lgb_probs >= 0.50).astype(int)

print("LightGBM at threshold 0.50:")
print("Confusion matrix (rows: actual, columns: predicted; Normal, Attack):")
print(confusion_matrix(y_val, lgb_default_preds, labels=[0, 1]))
print(classification_report(
    y_val,
    lgb_default_preds,
    target_names=["Normal", "Attack"],
))


LightGBM at threshold 0.50:
Confusion matrix (rows: actual, columns: predicted; Normal, Attack):
[[281930    899]
 [  1257  11177]]
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00    282829
      Attack       0.93      0.90      0.91     12434

    accuracy                           0.99    295263
   macro avg       0.96      0.95      0.95    295263
weighted avg       0.99      0.99      0.99    295263



### 11.4 Tuning Threshold Under the False Positive Limit

We are selecting the finite threshold with the highest validation recall subject to the same `TARGET_FPR` limit we used for logistic regression. 

You can see the validation metrics below.

In [29]:
lgb_fpr, lgb_tpr, lgb_thresholds = roc_curve(
    y_val,
    lgb_probs,
    drop_intermediate=False,
)

lgb_valid_indices = np.where(
    (lgb_fpr <= TARGET_FPR) & np.isfinite(lgb_thresholds)
)[0]
lgb_best_idx = lgb_valid_indices[np.argmax(lgb_tpr[lgb_valid_indices])]
lgb_threshold = lgb_thresholds[lgb_best_idx]
lgb_preds = (lgb_probs >= lgb_threshold).astype(int)

lgb_tn, lgb_fp, lgb_fn, lgb_tp = confusion_matrix(
    y_val, lgb_preds, labels=[0, 1]
).ravel()

print("Chosen threshold:", lgb_threshold)
print("TN:", lgb_tn)
print("FP:", lgb_fp)
print("FN:", lgb_fn)
print("TP:", lgb_tp)
print("Precision:", precision_score(y_val, lgb_preds))
print("Recall:", recall_score(y_val, lgb_preds))
print("F1:", f1_score(y_val, lgb_preds))
print("FPR:", lgb_fp / (lgb_fp + lgb_tn))
print("Average precision:", average_precision_score(y_val, lgb_probs))


Chosen threshold: 0.2395823380876071
TN: 280002
FP: 2827
FN: 247
TP: 12187
Precision: 0.8117090715332357
Recall: 0.9801351133987454
F1: 0.888006412124745
FPR: 0.009995438940136974
Average precision: 0.9803492834684779


### 11.5 Reviewing Missed Attacks by Family

Measuring detection and missed attacks for each family using a tuned LightGBM threshold.

In [30]:
lgb_audit = val_df[["Label", "attack_cat"]].copy()
lgb_audit["Predicted"] = lgb_preds
lgb_attacks = lgb_audit[lgb_audit["Label"] == 1].copy()

lgb_family_results = lgb_attacks.groupby("attack_cat").agg(
    Total=("Label", "size"),
    Detected=("Predicted", "sum"),
)
lgb_family_results["Missed"] = (
    lgb_family_results["Total"] - lgb_family_results["Detected"]
)
lgb_family_results["Recall"] = (
    lgb_family_results["Detected"] / lgb_family_results["Total"]
)

print(lgb_family_results.round(4).to_string())


                Total  Detected  Missed  Recall
attack_cat                                     
Analysis          272       229      43  0.8419
Backdoors         257       257       0  1.0000
Dos               789       788       1  0.9987
Exploits         4078      4068      10  0.9975
Fuzzers          3052      2863     189  0.9381
Generic          2384      2382       2  0.9992
Reconnaissance   1361      1360       1  0.9993
Shellcode         215       214       1  0.9953
Worms              26        26       0  1.0000


## 12. Tier 2: Detecting Attack Families

Training a mulitclass LightGBM model using only known attacks.

This evaluates family classification on actual attacks.

It does not yet measure the complete two-tier pipeline with flows passed through Tier 1.

### 12.1 Preparing Attack-Only Features and Targets

In [31]:
X_train_tier2_lgb = X_train_lgb.loc[attack_train.index].copy()
X_val_tier2_lgb = X_val_lgb.loc[attack_val.index].copy()

print("Tier 2 training:", X_train_tier2_lgb.shape)
print("Tier 2 validation:", X_val_tier2_lgb.shape)
print("Number of attack families:", y_train_tier2.nunique())


Tier 2 training: (58021, 36)
Tier 2 validation: (12434, 36)
Number of attack families: 9


### 12.2 Training the Model

In [32]:
tier2_model = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)
tier2_model.fit(
    X_train_tier2_lgb,
    y_train_tier2,
    categorical_feature=categorical_features,
)


,learning_rate,0.05
,n_estimators,300
,objective,'multiclass'
,random_state,42
,n_jobs,-1
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,class_weight,None


### 12.3 Evaluating Validation Predictions by Attack Family

In [33]:
tier2_predictions = tier2_model.predict(X_val_tier2_lgb)

print(classification_report(
    y_val_tier2,
    tier2_predictions,
    digits=4,
    zero_division=0,
))


                precision    recall  f1-score   support

      Analysis     0.1558    0.2206    0.1826       272
     Backdoors     0.1235    0.1556    0.1377       257
           Dos     0.4056    0.2560    0.3139       789
      Exploits     0.8269    0.8705    0.8482      4078
       Fuzzers     0.9038    0.9053    0.9046      3052
       Generic     0.9024    0.8683    0.8850      2384
Reconnaissance     0.7664    0.7546    0.7605      1361
     Shellcode     0.7064    0.7163    0.7113       215
         Worms     0.4800    0.4615    0.4706        26

      accuracy                         0.7944     12434
     macro avg     0.5857    0.5788    0.5794     12434
  weighted avg     0.7949    0.7944    0.7932     12434



### 12.4 Training with Balanced Class Weights

Train a second Tier 2 LightGBM model on the same attack-only training data, using balanced class weights to give rare families more influence.


In [34]:
tier2_balanced_model = lgb.LGBMClassifier(
    objective="multiclass",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)

tier2_balanced_model.fit(
    X_train_tier2_lgb,
    y_train_tier2,
    categorical_feature=categorical_features,
)


,learning_rate,0.05
,n_estimators,300
,objective,'multiclass'
,class_weight,'balanced'
,random_state,42
,n_jobs,-1
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000


### 12.5 Evaluating the Balanced Model

Evaluate the balanced model on the same attack-only validation set used for the original LightGBM model.


In [35]:
balanced_predictions = tier2_balanced_model.predict(
    X_val_tier2_lgb,
)

print(classification_report(
    y_val_tier2,
    balanced_predictions,
    digits=4,
    zero_division=0
))


                precision    recall  f1-score   support

      Analysis     0.1358    0.3713    0.1988       272
     Backdoors     0.1256    0.3930    0.1904       257
           Dos     0.4533    0.4246    0.4385       789
      Exploits     0.9126    0.7499    0.8233      4078
       Fuzzers     0.9548    0.8919    0.9222      3052
       Generic     0.9643    0.8716    0.9156      2384
Reconnaissance     0.7862    0.8266    0.8059      1361
     Shellcode     0.5891    0.9070    0.7143       215
         Worms     0.4643    0.5000    0.4815        26

      accuracy                         0.7824     12434
     macro avg     0.5984    0.6595    0.6101     12434
  weighted avg     0.8501    0.7824    0.8096     12434



### 12.6 Comparing LightGBM Models

Compare accuracy, macro F1, and weighted F1 to see how balancing affects overall performance and performance across families.


In [36]:
comparison = pd.DataFrame({
    "Model": [
        "Normal LightGBM",
        "Balanced LightGBM"
    ],
    "Accuracy": [
        accuracy_score(y_val_tier2, tier2_predictions),
        accuracy_score(y_val_tier2, balanced_predictions)
    ],
    "Macro F1": [
        f1_score(
            y_val_tier2,
            tier2_predictions,
            average="macro"
        ),
        f1_score(
            y_val_tier2,
            balanced_predictions,
            average="macro",
        )
    ],
    "Weighted F1": [
        f1_score(
            y_val_tier2,
            tier2_predictions,
            average="weighted"
        ),
        f1_score(
            y_val_tier2,
            balanced_predictions,
            average="weighted",
        )
    ],
})

print(comparison.round(4).to_string(index=False))


            Model  Accuracy  Macro F1  Weighted F1
  Normal LightGBM    0.7944    0.5794       0.7932
Balanced LightGBM    0.7824    0.6101       0.8096


### 12.7 Comparing LightGBM F1 by Attack Family

Compare each family's F1 score. Positive changes indicate an improvement from balanced class weights.


In [37]:
families = sorted(y_val_tier2.unique())

original_f1 = f1_score(
    y_val_tier2,
    tier2_predictions,
    labels=families,
    average=None,
    zero_division=0
)

balanced_f1 = f1_score(
    y_val_tier2,
    balanced_predictions,
    labels=families,
    average=None,
    zero_division=0
)

family_comparison = pd.DataFrame({
    "Attack Family": families,
    "Original F1": original_f1,
    "Balanced F1": balanced_f1
})

family_comparison["Change"] = (
    family_comparison["Balanced F1"]
    - family_comparison["Original F1"]
)

print(family_comparison.round(4).to_string(index=False))


 Attack Family  Original F1  Balanced F1  Change
      Analysis       0.1826       0.1988  0.0162
     Backdoors       0.1377       0.1904  0.0527
           Dos       0.3139       0.4385  0.1246
      Exploits       0.8482       0.8233 -0.0249
       Fuzzers       0.9046       0.9222  0.0177
       Generic       0.8850       0.9156  0.0306
Reconnaissance       0.7605       0.8059  0.0454
     Shellcode       0.7113       0.7143  0.0030
         Worms       0.4706       0.4815  0.0109


## 13. Tier 2: CatBoost

Train CatBoost on the same attack-only split using its native categorical features and `SqrtBalanced` class weights. This section requires `catboost` to be installed.

### 13.1 Preparing Features and Targets

Use the original attack-only features and convert categorical columns to strings for CatBoost.


In [38]:
from catboost import CatBoostClassifier


In [39]:
# Start with original attack-only features
X_train_cb = attack_train[feature_columns].copy()
X_val_cb = attack_val[feature_columns].copy()

# Our targets
y_train_cb = attack_train["attack_cat"]
y_val_cb = attack_val["attack_cat"]

# Convert categorical features to strings
for col in categorical_features:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_val_cb[col] = X_val_cb[col].astype(str)

# Check dimensions
print("Training:", X_train_cb.shape)
print("Validation:", X_val_cb.shape)
print("Attack families:", y_train_cb.nunique())


Training: (58021, 36)
Validation: (12434, 36)
Attack families: 9


### 13.2 Training CatBoost

Use the validation set for early stopping and retain the best iteration.


In [40]:
catboost_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3,
    auto_class_weights="SqrtBalanced",
    random_seed=42,
    thread_count=-1,
    verbose=100,
)

catboost_model.fit(
    X_train_cb,
    y_train_cb,
    cat_features=categorical_features,
    eval_set=(X_val_cb, y_val_cb),
    early_stopping_rounds=100,
    use_best_model=True,
)


0:	learn: 2.0463848	test: 2.0472048	best: 2.0472048 (0)	total: 216ms	remaining: 3m 36s
100:	learn: 0.8374428	test: 0.8506361	best: 0.8506361 (100)	total: 16.3s	remaining: 2m 25s
200:	learn: 0.7739090	test: 0.7942694	best: 0.7942694 (200)	total: 31.7s	remaining: 2m 6s
300:	learn: 0.7333672	test: 0.7611661	best: 0.7611661 (300)	total: 48s	remaining: 1m 51s
400:	learn: 0.7086914	test: 0.7447156	best: 0.7447074 (399)	total: 1m 4s	remaining: 1m 35s
500:	learn: 0.6882648	test: 0.7319687	best: 0.7319687 (500)	total: 1m 20s	remaining: 1m 20s
600:	learn: 0.6731111	test: 0.7253563	best: 0.7253563 (600)	total: 1m 35s	remaining: 1m 3s
700:	learn: 0.6587255	test: 0.7184198	best: 0.7183372 (692)	total: 1m 50s	remaining: 47.3s
800:	learn: 0.6463706	test: 0.7126027	best: 0.7126027 (800)	total: 2m 5s	remaining: 31.3s
900:	learn: 0.6347088	test: 0.7077135	best: 0.7077135 (900)	total: 2m 21s	remaining: 15.5s
999:	learn: 0.6242222	test: 0.7034315	best: 0.7034315 (999)	total: 2m 36s	remaining: 0us

bestTes

CatBoostClassifier(auto_class_weights='SqrtBalanced', depth=6, eval_metric='MultiClass', iterations=1000, l2_leaf_reg=3, learning_rate=0.05, loss_function='MultiClass', random_seed=42, verbose=100)

### 13.3 Inspecting the Best Iteration

Check which iteration CatBoost selected using validation multiclass loss.


In [41]:
print("Best iteration:", catboost_model.best_iteration_)


Best iteration: 999


### 13.4 Evaluating CatBoost

Evaluate family classification on the same attack-only validation rows as the LightGBM models. These results do not measure the complete two-tier pipeline.


In [42]:
catboost_predictions = np.asarray(
    catboost_model.predict(X_val_cb)
).ravel()

print("CatBoost Classification Report:")

print(classification_report(
    y_val_cb,
    catboost_predictions,
    digits=4,
    zero_division=0
))


CatBoost Classification Report:
                precision    recall  f1-score   support

      Analysis     0.2293    0.5294    0.3200       272
     Backdoors     0.1672    0.5603    0.2576       257
           Dos     0.5000    0.2586    0.3409       789
      Exploits     0.8479    0.7913    0.8186      4078
       Fuzzers     0.9215    0.8771    0.8988      3052
       Generic     0.9529    0.8142    0.8781      2384
Reconnaissance     0.8075    0.8046    0.8060      1361
     Shellcode     0.4633    0.8512    0.6000       215
         Worms     0.4211    0.6154    0.5000        26

      accuracy                         0.7746     12434
     macro avg     0.5901    0.6780    0.6022     12434
  weighted avg     0.8244    0.7746    0.7911     12434



## 14. Comparing Tier 2 Models

### 14.1 Comparing Overall Validation Metrics

Compare the original LightGBM, balanced LightGBM, and CatBoost models on the same validation targets. CatBoost also uses this validation set for early stopping and best-iteration selection, so these are model-selection results.


In [43]:
comparison = pd.DataFrame({
    "Model": [
        "Original LightGBM",
        "Balanced LightGBM",
        "CatBoost"
    ],

    "Accuracy": [
        accuracy_score(y_val_cb, tier2_predictions),
        accuracy_score(y_val_cb, balanced_predictions),
        accuracy_score(y_val_cb, catboost_predictions)
    ],

    "Macro F1": [
        f1_score(
            y_val_cb, tier2_predictions,
            average="macro"
        ),
        f1_score(
            y_val_cb, balanced_predictions,
            average="macro"
        ),
        f1_score(
            y_val_cb, catboost_predictions,
            average="macro",
        )
    ],

    "Weighted F1": [
        f1_score(
            y_val_cb, tier2_predictions,
            average="weighted"
        ),
        f1_score(
            y_val_cb, balanced_predictions,
            average="weighted"
        ),
        f1_score(
            y_val_cb, catboost_predictions,
            average="weighted",
        )
    ],
})

print(comparison.round(4).to_string(index=False))


            Model  Accuracy  Macro F1  Weighted F1
Original LightGBM    0.7944    0.5794       0.7932
Balanced LightGBM    0.7824    0.6101       0.8096
         CatBoost    0.7746    0.6022       0.7911


### 14.2 Comparing CatBoost and Balanced LightGBM by Family

Compare per-family F1 scores. Positive changes favor CatBoost; negative changes favor balanced LightGBM.


In [44]:
families = sorted(y_val_cb.unique())

catboost_f1 = f1_score(
    y_val_cb,
    catboost_predictions,
    labels=families,
    average=None,
    zero_division=0
)

balanced_f1 = f1_score(
    y_val_cb,
    balanced_predictions,
    labels=families,
    average=None,
    zero_division=0
)

family_comparison = pd.DataFrame({
    "Attack Family": families,
    "Balanced LightGBM F1": balanced_f1,
    "CatBoost F1": catboost_f1
})

family_comparison["Change"] = (
    family_comparison["CatBoost F1"]
    - family_comparison["Balanced LightGBM F1"]
)

print(family_comparison.round(4).to_string(index=False))


 Attack Family  Balanced LightGBM F1  CatBoost F1  Change
      Analysis                0.1988       0.3200  0.1212
     Backdoors                0.1904       0.2576  0.0672
           Dos                0.4385       0.3409 -0.0976
      Exploits                0.8233       0.8186 -0.0046
       Fuzzers                0.9222       0.8988 -0.0235
       Generic                0.9156       0.8781 -0.0375
Reconnaissance                0.8059       0.8060  0.0002
     Shellcode                0.7143       0.6000 -0.1143
         Worms                0.4815       0.5000  0.0185


## 15. Extended CatBoost

The remaining workflow follows four stages:

1. **Extended CatBoost experiment:** train for up to 2,000 iterations, evaluate attack-family predictions, and compare Tier 2 models.
2. **Full pipeline validation:** route flows through Tier 1 and Tier 2, inspect false alarms, and compare decision thresholds.
3. **Final test evaluation:** apply the selected threshold and balanced LightGBM pipeline to the held-out test data.
4. **Export:** save the selected models, inference metadata, and sample flows for the demo.


### 15.1 Training the Extended CatBoost Model


In [45]:
catboost_extended = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="MultiClass",
    iterations=2000,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=3,
    auto_class_weights="SqrtBalanced",
    random_seed=42,
    thread_count=-1,
    verbose=200
)

catboost_extended.fit(
    X_train_cb,
    y_train_cb,
    cat_features=categorical_features,
    eval_set=(X_val_cb, y_val_cb),
    early_stopping_rounds=150,
    use_best_model=True
)

print("Best iteration:", catboost_extended.best_iteration_)


0:	learn: 2.0463848	test: 2.0472048	best: 2.0472048 (0)	total: 130ms	remaining: 4m 19s
200:	learn: 0.7739090	test: 0.7942694	best: 0.7942694 (200)	total: 29.4s	remaining: 4m 23s
400:	learn: 0.7086914	test: 0.7447156	best: 0.7447074 (399)	total: 1m	remaining: 3m 59s
600:	learn: 0.6731111	test: 0.7253563	best: 0.7253563 (600)	total: 1m 30s	remaining: 3m 29s
800:	learn: 0.6463706	test: 0.7126027	best: 0.7126027 (800)	total: 2m	remaining: 3m 1s
1000:	learn: 0.6241771	test: 0.7034177	best: 0.7034177 (1000)	total: 2m 33s	remaining: 2m 32s
1200:	learn: 0.6046959	test: 0.6974697	best: 0.6974671 (1199)	total: 3m 4s	remaining: 2m 3s
1400:	learn: 0.5880372	test: 0.6934463	best: 0.6934067 (1396)	total: 3m 36s	remaining: 1m 32s
1600:	learn: 0.5736308	test: 0.6905800	best: 0.6905775 (1599)	total: 4m 7s	remaining: 1m 1s
1800:	learn: 0.5597046	test: 0.6883458	best: 0.6882573 (1790)	total: 4m 39s	remaining: 30.9s
1999:	learn: 0.5466316	test: 0.6865171	best: 0.6865171 (1999)	total: 5m 10s	remaining: 0us

### 15.2 Evaluation

In [46]:
extended_predictions = (
    catboost_extended.predict(X_val_cb)
    .ravel()
)

print("Extended CatBoost Classification Report:")

print(classification_report(
    y_val_cb,
    extended_predictions,
    digits=4,
    zero_division=0
))


Extended CatBoost Classification Report:
                precision    recall  f1-score   support

      Analysis     0.2206    0.4890    0.3040       272
     Backdoors     0.1650    0.5058    0.2488       257
           Dos     0.4860    0.2864    0.3604       789
      Exploits     0.8553    0.8028    0.8282      4078
       Fuzzers     0.9286    0.8824    0.9049      3052
       Generic     0.9525    0.8251    0.8842      2384
Reconnaissance     0.8018    0.8024    0.8021      1361
     Shellcode     0.4791    0.8512    0.6131       215
         Worms     0.4390    0.6923    0.5373        26

      accuracy                         0.7814     12434
     macro avg     0.5920    0.6819    0.6092     12434
  weighted avg     0.8271    0.7814    0.7975     12434



### 15.3 Comparing Models

In [47]:
comparison = pd.DataFrame({
    "Model": [
        "Original LightGBM",
        "Balanced LightGBM",
        "CatBoost (1000)",
        "CatBoost (2000 max)"
    ],

    "Macro F1": [
        f1_score(y_val_cb, tier2_predictions, average="macro"),
        f1_score(y_val_cb, balanced_predictions, average="macro"),
        f1_score(y_val_cb, catboost_predictions, average="macro"),
        f1_score(y_val_cb, extended_predictions, average="macro")
    ],

    "Weighted F1": [
        f1_score(y_val_cb, tier2_predictions, average="weighted"),
        f1_score(y_val_cb, balanced_predictions, average="weighted"),
        f1_score(y_val_cb, catboost_predictions, average="weighted"),
        f1_score(y_val_cb, extended_predictions, average="weighted")
    ],

    "Accuracy": [
        accuracy_score(y_val_cb, tier2_predictions),
        accuracy_score(y_val_cb, balanced_predictions),
        accuracy_score(y_val_cb, catboost_predictions),
        accuracy_score(y_val_cb, extended_predictions)
    ]
})

print(comparison.round(4).to_string(index=False))


              Model  Macro F1  Weighted F1  Accuracy
  Original LightGBM    0.5794       0.7932    0.7944
  Balanced LightGBM    0.6101       0.8096    0.7824
    CatBoost (1000)    0.6022       0.7911    0.7746
CatBoost (2000 max)    0.6092       0.7975    0.7814


## 16. Full Pipeline

### 16.1 Get Tier 1 Decisions

In [48]:
# Use the previously selected Tier 1 threshold
tier1_mask = lgb_probs >= lgb_threshold

# Indices of flows Tier 1 thinks are attacks
attack_indices = val_df.index[tier1_mask]

print("Flows passed to Tier 2:", len(attack_indices))
print("Flows predicted normal:", (~tier1_mask).sum())


Flows passed to Tier 2: 15014
Flows predicted normal: 280249


### 16.2 Building both complete Pipelines

In [49]:
# Every flow initially receives "Normal"
pipeline_lgb = pd.Series("Normal", index=val_df.index, dtype=object)

pipeline_cat = pd.Series("Normal", index=val_df.index, dtype=object)

# Balanced LightGBM Tier 2 predictions
pipeline_lgb.loc[attack_indices] = (
    tier2_balanced_model.predict(X_val_lgb.loc[attack_indices])
)

# Prepare the same flagged flows for CatBoost
X_flagged_cb = val_df.loc[attack_indices, feature_columns].copy()

for col in categorical_features:
    X_flagged_cb[col] = X_flagged_cb[col].astype(str)

# Extended CatBoost Tier 2 predictions
pipeline_cat.loc[attack_indices] = (catboost_extended.predict(X_flagged_cb).ravel())


### 16.3 Evaluating the Complete Pipelines


In [50]:
y_true_full = val_df["attack_cat"]

families = sorted(y_true_full[y_true_full != "Normal"].unique())

for name, predictions in [
    ("LightGBM Pipeline", pipeline_lgb),
    ("CatBoost Pipeline", pipeline_cat)
]:
    print("\n", name)

    print(classification_report(
        y_true_full,
        predictions,
        labels=["Normal"] + families,
        digits=4,
        zero_division=0
    ))

    print(
        "Attack-family Macro F1:",
        f1_score(
            y_true_full,
            predictions,
            labels=families,
            average="macro",
            zero_division=0
        )
    )



 LightGBM Pipeline
                precision    recall  f1-score   support

        Normal     0.9991    0.9900    0.9945    282829
      Analysis     0.0736    0.2169    0.1099       272
     Backdoors     0.1253    0.3930    0.1900       257
           Dos     0.4198    0.4246    0.4222       789
      Exploits     0.8721    0.7494    0.8061      4078
       Fuzzers     0.4999    0.8309    0.6242      3052
       Generic     0.9620    0.8716    0.9146      2384
Reconnaissance     0.7796    0.8266    0.8024      1361
     Shellcode     0.4887    0.9070    0.6352       215
         Worms     0.4483    0.5000    0.4727        26

      accuracy                         0.9805    295263
     macro avg     0.5669    0.6710    0.5972    295263
  weighted avg     0.9873    0.9805    0.9832    295263

Attack-family Macro F1: 0.5530431336078423

 CatBoost Pipeline
                precision    recall  f1-score   support

        Normal     0.9991    0.9900    0.9945    282829
      Analysis   

### 16.4 Inspecting Pipeline False Alarms


In [51]:
# Find actual normal traffic
normal_mask = val_df["Label"] == 0

print("LightGBM pipeline false alarms:")
print(
    pipeline_lgb[normal_mask]
    .value_counts()
    .drop("Normal", errors="ignore")
)

print("\nCatBoost pipeline false alarms:")
print(
    pipeline_cat[normal_mask]
    .value_counts()
    .drop("Normal", errors="ignore")
)


LightGBM pipeline false alarms:
Fuzzers           2412
Exploits           156
Analysis           111
Shellcode           68
Dos                 60
Reconnaissance      12
Generic              5
Backdoors            2
Worms                1
Name: count, dtype: int64

CatBoost pipeline false alarms:
Fuzzers           2378
Exploits           202
Analysis           116
Shellcode           92
Dos                 18
Reconnaissance      15
Generic              4
Backdoors            2
Name: count, dtype: int64


In [52]:
false_alarm_comparison = pd.DataFrame({
    "LightGBM": (
        pipeline_lgb[normal_mask]
        .value_counts()
    ),
    "CatBoost": (
        pipeline_cat[normal_mask]
        .value_counts()
    )
}).fillna(0)

# Remove correctly predicted normal traffic
false_alarm_comparison = false_alarm_comparison.drop(
    "Normal",
    errors="ignore"
)

print(false_alarm_comparison.astype(int).to_string())


                LightGBM  CatBoost
Analysis             111       116
Backdoors              2         2
Dos                   60        18
Exploits             156       202
Fuzzers             2412      2378
Generic                5         4
Reconnaissance        12        15
Shellcode             68        92
Worms                  1         0


### 16.5 Comparing Pipeline Thresholds


In [53]:
threshold_results = []

y_true = val_df["attack_cat"]
actual_binary = val_df["Label"]

families = sorted(y_true[y_true != "Normal"].unique())

for threshold in [lgb_threshold, 0.35, 0.50]:
    for model_name, base_predictions in [
        ("LightGBM", pipeline_lgb),
        ("CatBoost", pipeline_cat)
    ]:
        # Reuse predictions at the original lower threshold
        predictions = base_predictions.copy()

        # Higher thresholds reject more flows as Normal
        predictions.loc[val_df.index[lgb_probs < threshold]] = "Normal"

        predicted_binary = (predictions != "Normal").astype(int)

        fp = (
            (actual_binary == 0)
            & (predicted_binary == 1)
        ).sum()

        fn = (
            (actual_binary == 1)
            & (predicted_binary == 0)
        ).sum()

        normal_count = (actual_binary == 0).sum()

        threshold_results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Accuracy": accuracy_score(y_true, predictions),
            "Attack Macro F1": f1_score(
                y_true,
                predictions,
                labels=families,
                average="macro",
                zero_division=0
            ),
            "Tier 1 Recall": recall_score(
                actual_binary,
                predicted_binary
            ),
            "FPR": fp / normal_count,
            "False Alarms": fp,
            "Missed Attacks": fn
        })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4).to_string(index=False))


   Model  Threshold  Accuracy  Attack Macro F1  Tier 1 Recall    FPR  False Alarms  Missed Attacks
LightGBM     0.2396    0.9805           0.5530         0.9801 0.0100          2827             247
CatBoost     0.2396    0.9804           0.5521         0.9801 0.0100          2827             247
LightGBM     0.3500    0.9827           0.5550         0.9546 0.0067          1890             564
CatBoost     0.3500    0.9826           0.5539         0.9546 0.0067          1890             564
LightGBM     0.5000    0.9839           0.5529         0.8989 0.0032           899            1257
CatBoost     0.5000    0.9838           0.5527         0.8989 0.0032           899            1257


## 17. Final Test Evaluation

### 17.1 Preparing Test Features


In [54]:
FINAL_THRESHOLD = 0.35

X_test_final = test_df[feature_columns].copy()

for col in categorical_features:
    X_test_final[col] = pd.Categorical(
        X_test_final[col].astype(str),
        categories=X_train_lgb[col].cat.categories
    )

print("Test data:", X_test_final.shape)


Test data: (295263, 36)


### 17.2 Generating Final Pipeline Predictions


In [55]:
# Tier 1 probabilities
test_probs = lgb_model.predict_proba(X_test_final)[:, 1]

# Tier 1 decisions
test_attack_mask = test_probs >= FINAL_THRESHOLD

# Default all predictions to Normal
final_predictions = pd.Series(
    "Normal",
    index=test_df.index,
    dtype=object
)

# Only suspicious flows reach Tier 2
flagged_indices = test_df.index[test_attack_mask]

if len(flagged_indices) > 0:
    final_predictions.loc[flagged_indices] = (
        np.asarray(
            tier2_balanced_model.predict(X_test_final.loc[flagged_indices])
        ).ravel()
    )

print("Flows passed to Tier 2:", test_attack_mask.sum())
print("Predicted normal:", (~test_attack_mask).sum())


Flows passed to Tier 2: 13760
Predicted normal: 281503


### 17.3 Evaluating Tier 1 on Test Data


In [56]:
y_test_binary = test_df["Label"]

tn, fp, fn, tp = confusion_matrix(
    y_test_binary,
    test_attack_mask.astype(int),
    labels=[0, 1]
).ravel()

print("FINAL TIER 1 TEST RESULTS")
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

print("Precision:", precision_score(y_test_binary, test_attack_mask))

print("Recall:", recall_score(y_test_binary, test_attack_mask))

print("F1:", f1_score(y_test_binary, test_attack_mask))

print("FPR:", fp / (fp + tn))

print("Average Precision:", average_precision_score(y_test_binary, test_probs))


FINAL TIER 1 TEST RESULTS
TN: 280910
FP: 1920
FN: 593
TP: 11840
Precision: 0.8604651162790697
Recall: 0.9523043513230918
F1: 0.9040583361966937
FPR: 0.006788530212495139
Average Precision: 0.9794375634525715


### 17.4 Evaluating the Complete Pipeline on Test Data


In [57]:
y_test_full = test_df["attack_cat"]

families = sorted(train_df.loc[train_df["Label"] == 1, "attack_cat"].unique())

print("FINAL END-TO-END TEST RESULTS")

print(classification_report(
    y_test_full,
    final_predictions,
    labels=["Normal"] + families,
    digits=4,
    zero_division=0
))

print("Attack-family Macro F1:",
    f1_score(
        y_test_full,
        final_predictions,
        labels=families,
        average="macro",
        zero_division=0
    )
)

print("Overall Accuracy:",
    accuracy_score(y_test_full, final_predictions)
)


FINAL END-TO-END TEST RESULTS
                precision    recall  f1-score   support

        Normal     0.9979    0.9932    0.9955    282830
      Analysis     0.0648    0.1771    0.0949       271
     Backdoors     0.1079    0.3398    0.1638       256
           Dos     0.3929    0.4367    0.4137       790
      Exploits     0.8827    0.7268    0.7972      4078
       Fuzzers     0.5656    0.7343    0.6390      3052
       Generic     0.9537    0.8637    0.9064      2384
Reconnaissance     0.7782    0.8091    0.7934      1362
     Shellcode     0.4818    0.9256    0.6338       215
         Worms     0.6296    0.6800    0.6538        25

      accuracy                         0.9821    295263
     macro avg     0.5855    0.6686    0.6092    295263
  weighted avg     0.9868    0.9821    0.9841    295263

Attack-family Macro F1: 0.566223684117274
Overall Accuracy: 0.9820803825741796


### 17.5 Exporting selected models and evaluation inputs

In [58]:
from pathlib import Path
import joblib

# Run from notebooks/, matching the launch command above.
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

metadata = {
    "feature_columns": list(feature_columns),
    "categorical_features": list(categorical_features),
    "categorical_levels": {
        col: X_train_lgb[col].cat.categories.tolist()
        for col in categorical_features
    },
    "threshold": float(FINAL_THRESHOLD),
}

joblib.dump(lgb_model, model_dir / "tier1_lightgbm.joblib")
joblib.dump(tier2_balanced_model, model_dir / "tier2_lightgbm.joblib")
joblib.dump(metadata, model_dir / "metadata.joblib")

test_df.to_csv("../data/UNSW-NB15/test.csv", index=False)
test_df.sample(n=100, random_state=42).to_csv(
    "../data/UNSW-NB15/demo_mixed_flows.csv", index=False
)
